# Data Agent Platform Studio — Colab GPU / CPU

A self-contained Data Agent Studio using one shared Bronze/Silver/Gold pipeline and two CrewAI agents. Both agents share one local Hugging Face model. CUDA (T4 target) is recommended; CPU-only inference is also supported (slower, FP32).

No external hosted-agent or vendor-agent API is used. Select a GPU runtime when available, or use the standard CPU runtime. Set `HF_DEVICE=cpu` to force CPU or `HF_DEVICE=auto` to prefer CUDA and fall back to CPU.

**Important:** when a notebook is already open, `git pull` updates repository files on disk but does not refresh code cells already loaded in the browser tab. After a notebook update, close the old Colab tab and reopen this notebook from the repository link before running cells.

In [ ]:
%cd /content
!if [ -d data-agent-platform-studio/.git ]; then cd data-agent-platform-studio && git fetch origin feature/colab-crewai-dual-agent-studio && git checkout feature/colab-crewai-dual-agent-studio && git pull --ff-only origin feature/colab-crewai-dual-agent-studio; else git clone -b feature/colab-crewai-dual-agent-studio https://github.com/msellamiTN/data-agent-platform-studio.git; fi
%cd /content/data-agent-platform-studio
!git rev-parse --short HEAD

In [ ]:
from pathlib import Path
import subprocess
import sys

bootstrap_file = Path("scripts/bootstrap_colab.py")
if not bootstrap_file.is_file():
    raise FileNotFoundError(
        "scripts/bootstrap_colab.py is missing. Re-run the repository checkout cell."
    )

bootstrap_source = bootstrap_file.read_text(encoding="utf-8")
if 'uv, "venv"' not in bootstrap_source or '"-m", "venv"' in bootstrap_source:
    raise RuntimeError(
        "An outdated Colab bootstrap script was checked out. Close this notebook tab, "
        "reopen the current repository notebook, and run the checkout cell again."
    )

print("Data2AI Colab bootstrap: uv-based, ensurepip-free", flush=True)
# Use the exact interpreter running the notebook kernel (the Colab Torch runtime).
subprocess.check_call([sys.executable, "scripts/bootstrap_colab.py"])


In [ ]:
import os, platform, torch
from packaging.version import Version

print("Notebook Python:", platform.python_version())
print("Colab base PyTorch:", torch.__version__)
if Version(torch.__version__.split("+")[0]) < Version("2.6.0"):
    raise RuntimeError(
        "PyTorch >= 2.6 is required. Select a current Colab runtime; "
        "the notebook will not replace Colab's CUDA-specific PyTorch."
    )
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
    os.environ.setdefault("HF_DEVICE", "auto")
else:
    os.environ["HF_DEVICE"] = "cpu"
    print("Using CPU inference (FP32). Expect slower generation and higher RAM use.")
    print("CPU threads:", os.getenv("HF_CPU_THREADS", "4"))


In [ ]:
import subprocess
from pathlib import Path

venv_python = Path(".venv/bin/python").resolve()
subprocess.check_call([
    str(venv_python), "-c",
    "from studio.config import MODEL_ID; import os; "
    "print('Hugging Face model:', MODEL_ID); "
    "print('Selected device:', os.getenv('HF_DEVICE', 'auto')); "
    "print('Precision: FP16 on CUDA; FP32 on CPU')",
])


## Live runtime acceptance and launch

The isolated Studio process loads the local model, runs one inference, and requires one real CrewAI invocation of a registered Python tool. If both checks pass, it launches Gradio in the same process using the already-loaded model. This avoids a second model download/load and keeps Colab's unrelated preinstalled packages untouched.

The launch URL is public to anyone who obtains it. Use synthetic data only.

In [ ]:
import subprocess
from pathlib import Path
import sys

venv_python = Path(".venv/bin/python").resolve()
if not venv_python.is_file():
    raise RuntimeError(
        f"Isolated Python was not created at {venv_python}. "
        "Inspect the bootstrap cell output before launching."
    )

# Never use shell "python" here: it may resolve to a different installation.
subprocess.check_call([
    str(venv_python),
    "scripts/launch_colab.py",
])
